In [1]:
#@title Mount Goole Drive

import os
import sys
from pathlib import Path

root_path = "/content/drive/MyDrive/MSc/Flood-Mapping"  #@param {type:"string", multiline:true}

mount_drive = True  #@param {type:"boolean"}

clone_repo = False  #@param {type:"boolean"}

if clone_repo and mount_drive:

    from google.colab import drive
    drive.mount("/content/drive")

    root_path = os.path.join(root_path, "Flood-Mapping")

    !git clone https://github.com/TAX2310/Flood-Mapping.git $root_path

    sys.path.append(os.path.join(root_path))

    from S1.s1_config import S1_CFG
    cfg = S1_CFG()

    cfg.ROOT = Path(root_path)

elif not clone_repo and mount_drive:
    from google.colab import drive
    drive.mount("/content/drive")

    sys.path.append(os.path.join(root_path))

    from S1.s1_config import S1_CFG
    cfg = S1_CFG()

    cfg.ROOT = Path(root_path)

elif clone_repo and not mount_drive:
    root_path = "Flood-Mapping"

    !git clone https://github.com/TAX2310/Flood-Mapping.git

    sys.path.append(root_path)

    from S1.s1_config import S1_CFG
    cfg = S1_CFG()

    cfg.ROOT = Path(root_path)

else:
    from S1.s1_config import S1_CFG
    cfg = S1_CFG()

    sys.path.append(os.path.join(cfg.ROOT))

Mounted at /content/drive


In [14]:
from S1.data.dataloader import *
from S1 import *

In [15]:
temp = get_intersection_ids("/content/drive/MyDrive/MSc/Flood-Mapping/Dataset/Sentinel1_metadata.csv", "/content/drive/MyDrive/MSc/Flood-Mapping/Dataset/Sentinel2_metadata.csv")

In [16]:
print(len(temp))
print(temp)

271
{'EMSR419_AOI04_07_09_2_1', 'EMSR292_01CHRISOUPOLI_06_04_2_1', 'EMSR467_AOI03_10_02_2_1', 'EMSR629_AOI01_08_02_1_2', 'EMSR501_AOI01_10_05_2_1', 'EMSR419_AOI04_01_08_2_1', 'EMSR419_AOI05_03_10_2_2', 'EMSR419_AOI05_03_09_2_1', 'EMSR501_AOI01_09_05_2_2', 'EMSR292_01CHRISOUPOLI_11_05_2_1', 'EMSR501_AOI01_06_07_2_2', 'EMSR467_AOI03_08_04_2_1', 'EMSR419_AOI05_04_09_1_1', 'EMSR419_AOI05_03_09_1_1', 'EMSR292_01CHRISOUPOLI_11_03_1_2', 'EMSR292_01CHRISOUPOLI_11_05_2_2', 'EMSR501_AOI01_05_07_2_1', 'EMSR629_AOI01_07_01_2_2', 'EMSR629_AOI02_01_13_1_1', 'EMSR501_AOI01_05_07_1_2', 'EMSR419_AOI03_3_4_2_1', 'EMSR292_01CHRISOUPOLI_05_07_2_1', 'EMSR501_AOI01_08_03_1_1', 'EMSR629_AOI01_07_02_2_1', 'EMSR419_AOI03_3_5_2_2', 'EMSR419_AOI04_03_03_1_1', 'EMSR419_AOI04_03_06_2_1', 'EMSR419_AOI04_03_03_2_2', 'EMSR467_AOI02_10_02_1_2', 'EMSR292_01CHRISOUPOLI_07_05_1_2', 'EMSR419_AOI04_02_02_1_2', 'EMSR292_01CHRISOUPOLI_11_03_1_1', 'EMSR292_01CHRISOUPOLI_06_07_1_2', 'EMSR292_01CHRISOUPOLI_08_02_2_1', 'EMSR419_

In [30]:
from pathlib import Path
import pandas as pd


def load_metadata(csv_path):
    """
    Load a metadata CSV and remove any unnamed index columns.
    """
    df = pd.read_csv(csv_path)
    df = df.loc[:, ~df.columns.str.contains(r"^Unnamed")]
    return df

def get_valid_matching_tile_ids(
    s1_csv_path,
    s2_csv_path,
    floodmap_key="floodmap_id",
    tile_key="tile_id",
    floodmap_date_col="floodmap_date",
    sentinel_time_col="sentinel_timestamp",
    max_time_diff_hours=24,
    require_matching_floodmap_id=True,
    strip_suffix=True,
):
    """
    Return tile_ids that exist in both S1 and S2, where each row is internally
    valid based on the time difference between floodmap_date and sentinel_timestamp.

    Parameters
    ----------
    s1_csv_path, s2_csv_path : str or Path
        Paths to the S1 and S2 metadata CSVs.
    floodmap_key : str
        Column used for floodmap identifier.
    tile_key : str
        Column used for tile identifier.
    floodmap_date_col : str
        Column containing flood map timestamp.
    sentinel_time_col : str
        Column containing image acquisition timestamp.
    max_time_diff_hours : float
        Maximum allowed absolute difference in hours between floodmap_date and
        sentinel_timestamp within each dataset.
    require_matching_floodmap_id : bool
        If True, require matches on both floodmap_id and tile_id.
        If False, match only on tile_id after temporal filtering.
    strip_suffix : bool
        If True, remove file suffix using Path.stem.

    Returns
    -------
    set[str]
        Matching tile_ids that pass the temporal validity filter.
    """

    s1_df = load_metadata(s1_csv_path).copy()
    s2_df = load_metadata(s2_csv_path).copy()

    # Parse timestamps
    s1_df[floodmap_date_col] = pd.to_datetime(s1_df[floodmap_date_col], errors="coerce")
    s1_df[sentinel_time_col] = pd.to_datetime(s1_df["sentinel_date"], errors="coerce")

    s2_df[floodmap_date_col] = pd.to_datetime(s2_df[floodmap_date_col], errors="coerce")
    s2_df[sentinel_time_col] = pd.to_datetime(s2_df[sentinel_time_col], errors="coerce")

    # Compute absolute time difference in hours within each dataset
    s1_df["time_diff_hours"] = (
        (s1_df[floodmap_date_col] - s1_df[sentinel_time_col]).abs().dt.total_seconds() / 3600
    )
    s2_df["time_diff_hours"] = (
        (s2_df[floodmap_date_col] - s2_df[sentinel_time_col]).abs().dt.total_seconds() / 3600
    )

    # Keep only rows within acceptable internal range
    s1_valid = s1_df[s1_df["time_diff_hours"] <= max_time_diff_hours].copy()
    s2_valid = s2_df[s2_df["time_diff_hours"] <= max_time_diff_hours].copy()

    # Normalise join columns as strings
    s1_valid[tile_key] = s1_valid[tile_key].astype(str)
    s2_valid[tile_key] = s2_valid[tile_key].astype(str)

    if require_matching_floodmap_id:
        s1_valid[floodmap_key] = s1_valid[floodmap_key].astype(str)
        s2_valid[floodmap_key] = s2_valid[floodmap_key].astype(str)

        merged = s1_valid.merge(
            s2_valid,
            on=[floodmap_key, tile_key],
            suffixes=("_s1", "_s2"),
        )
    else:
        merged = s1_valid.merge(
            s2_valid,
            on=[tile_key],
            suffixes=("_s1", "_s2"),
        )

    tile_ids = set(merged[tile_key])

    if strip_suffix:
        tile_ids = {Path(x).stem for x in tile_ids}

    return tile_ids


In [31]:
valid_tiles = get_valid_matching_tile_ids(
    s1_csv_path="/content/drive/MyDrive/MSc/Flood-Mapping/Dataset/Sentinel1_metadata.csv",
    s2_csv_path="/content/drive/MyDrive/MSc/Flood-Mapping/Dataset/Sentinel2_metadata.csv",
    max_time_diff_hours=24,
    require_matching_floodmap_id=True,
)

print(len(valid_tiles))
print(valid_tiles)

84
{'EMSR419_AOI04_07_09_2_1', 'EMSR419_AOI05_03_10_2_2', 'EMSR419_AOI05_03_09_2_1', 'EMSR419_AOI04_01_08_2_1', 'EMSR419_AOI05_04_09_1_1', 'EMSR419_AOI05_03_09_1_1', 'EMSR629_AOI02_01_13_1_1', 'EMSR419_AOI03_3_4_2_1', 'EMSR419_AOI03_3_5_2_2', 'EMSR419_AOI04_03_03_1_1', 'EMSR419_AOI04_03_06_2_1', 'EMSR419_AOI04_03_03_2_2', 'EMSR419_AOI04_02_02_1_2', 'EMSR419_AOI05_02_10_2_1', 'EMSR419_AOI04_02_06_1_1', 'EMSR419_AOI04_04_04_2_2', 'EMSR419_AOI05_03_10_1_2', 'EMSR629_AOI02_01_09_1_2', 'EMSR419_AOI03_2_5_2_1', 'EMSR419_AOI04_03_04_2_2', 'EMSR419_AOI05_04_03_1_1', 'EMSR419_AOI04_04_04_1_1', 'EMSR629_AOI02_01_13_2_2', 'EMSR419_AOI05_03_09_2_2', 'EMSR629_AOI02_01_11_2_2', 'EMSR419_AOI05_02_10_2_2', 'EMSR419_AOI04_04_04_1_2', 'EMSR419_AOI03_3_4_2_2', 'EMSR419_AOI03_2_6_1_2', 'EMSR419_AOI04_01_07_2_2', 'EMSR419_AOI04_04_06_2_2', 'EMSR419_AOI04_02_02_2_1', 'EMSR419_AOI04_03_04_1_2', 'EMSR419_AOI05_03_10_2_1', 'EMSR419_AOI05_02_08_2_2', 'EMSR419_AOI04_03_04_1_1', 'EMSR419_AOI04_07_08_2_2', 'EMSR41

In [ ]:
train_loader, val_loader, test_loader = make_dataloaders(cfg)

batch = next(iter(train_loader))
print(batch["image"].shape)   # [B, 2, 128, 128]
print(batch["mask"].shape)    # [B, 128, 128]
print(batch["image"].dtype, batch["mask"].dtype)
print(batch["id"][:3])